# 01 — Building Versioned RAG Evaluation Datasets


## Mission and learning contract

You own evaluation data for Northstar's internal policy assistant. Build a release-grade dataset—not a list of convenient questions. The corpus contains current and historical policies, multi-evidence answers, distractors, restricted material, tenant boundaries, and unanswerable requests.

**Workflow:** define → generate candidates → mutate → validate → review → promote → version. Synthetic generation proposes coverage; a golden case requires review.


In [ ]:
from collections import Counter
import json, os, re, time
from pathlib import Path
import pandas as pd

from evaluation_contracts import *

pd.set_option("display.max_colwidth", 90)
corpus = load_corpus()
golden = load_cases()
print(f"Loaded {len(corpus)} corpus chunks and {len(golden)} golden cases.")


## 1. Inspect the corpus and its failure opportunities

Stable document and chunk IDs let retrieval metrics survive text edits. No credentials or secrets belong in this corpus.


In [ ]:
corpus_df = pd.DataFrame([c.model_dump() for c in corpus])
display(corpus_df.groupby(["classification", "status", "tenant_id"]).size().rename("chunks"))
display(corpus_df[["chunk_id", "status", "classification", "text"]].head(8))
assert not corpus_df.text.str.contains(r"password|api[_ -]?key|private key|access token", case=False, regex=True).any()


## 2. One shared, typed case contract

`EvalCase` separates required evidence from merely relevant evidence. That distinction is necessary for multi-evidence completeness. `review_status` prevents synthetic candidates from silently becoming release gates.


In [ ]:
schema_fields = list(EvalCase.model_json_schema()["properties"])
print(schema_fields)
display(pd.DataFrame([c.model_dump() for c in golden]).head(4))
assert validate_dataset(golden, corpus) == []
assert {c.slice for c in golden} >= {"unanswerable", "multi_evidence", "tenant_boundary", "stale_source"}


## 3. Candidate generation: optional real model, committed offline artifacts

The notebook never requires a paid API. Set `RUN_REAL_GENERATION=1`, `OPENAI_API_KEY`, and optionally `MODEL_NAME` to try schema-constrained generation. Generated output still enters `synthetic_unreviewed`.


In [ ]:
from pydantic import BaseModel, Field

class CandidateDraft(BaseModel):
    query: str
    answerable: bool
    expected_document_ids: list[str] = Field(default_factory=list)
    required_evidence_ids: list[str] = Field(default_factory=list)
    relevant_evidence_ids: list[str] = Field(default_factory=list)
    reference_answer: str | None = None
    slice: str
    risk: str

def optional_generator():
    if os.getenv("RUN_REAL_GENERATION") != "1":
        return None
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("RUN_REAL_GENERATION=1 requires OPENAI_API_KEY")
    from langchain_openai import ChatOpenAI
    model = os.getenv("MODEL_NAME", "gpt-4o-mini")
    return ChatOpenAI(model=model, temperature=0).with_structured_output(CandidateDraft, method="json_schema")

generator = optional_generator()
print("Real generator enabled:" if generator else "Offline mode: using committed candidate artifacts.", bool(generator))


In [ ]:
candidate_rows = load_json("evaluation_candidates.json")
print(f"Persisted candidates: {len(candidate_rows)}")
display(pd.DataFrame(candidate_rows)[["case_id", "slice", "risk", "review_status"]].head())

# Example only; intentionally not run offline.
generation_instruction = """Create one evaluation candidate grounded only in the supplied chunks.
Return stable evidence IDs. Mark unsupported questions unanswerable. Do not invent a reference answer."""
if generator:
    sample_chunks = [c.model_dump() for c in corpus[:3]]
    live_draft = generator.invoke(generation_instruction + "\n" + json.dumps(sample_chunks))
    print(live_draft.model_dump())


## 4. Mutations must preserve semantic validity

Changing the question without recomputing answerability, evidence, and reference answer creates invalid labels. Below, one mutation changes the supported answer and another becomes unanswerable.


In [ ]:
def mutate_case(source: EvalCase, *, case_id: str, query: str, answerable: bool,
                required: list[str], relevant: list[str], reference: str | None, slice_: str) -> EvalCase:
    chunk_to_document = {chunk.chunk_id: chunk.document_id for chunk in corpus}
    return EvalCase.model_validate({**source.model_dump(),
        "case_id": case_id, "query": query, "answerable": answerable,
        "expected_document_ids": sorted({chunk_to_document[item] for item in relevant}),
        "required_evidence_ids": required, "relevant_evidence_ids": relevant,
        "reference_answer": reference, "slice": slice_, "review_status": "synthetic_unreviewed",
        "reviewer_rationale": "Mutation recomputed against the current corpus; review still required.",
    })

base = golden[1]
changed_answer = mutate_case(base, case_id="mutation-current", query="What is the current carryover limit?",
    answerable=True, required=["leave-policy-v2#carryover"],
    relevant=["leave-policy-v2#carryover", "leave-policy-v1#carryover"],
    reference="The current limit is five days, expiring March 31.", slice_="explicit_freshness")
unanswerable = mutate_case(base, case_id="mutation-unsupported", query="Who approved my personal leave request?",
    answerable=False, required=[], relevant=[], reference=None, slice_="unanswerable")
display(pd.DataFrame([changed_answer.model_dump(), unanswerable.model_dump()])[["query","answerable","required_evidence_ids","reference_answer"]])


## 5. Validate before review

Schema validation catches per-case contradictions. Dataset validation catches duplicate IDs and evidence IDs absent from the corpus.


In [ ]:
from pydantic import ValidationError

valid, rejected = [], []
for row in candidate_rows:
    try:
        valid.append(EvalCase.model_validate(row))
    except ValidationError as exc:
        rejected.append({"case_id": row.get("case_id"), "reason": str(exc).splitlines()[0]})

dataset_errors = validate_dataset(valid, corpus)
print("Schema rejects:", rejected)
print("Dataset errors:", dataset_errors)
assert rejected and dataset_errors  # deliberately bad candidates are visible, not silently repaired
assert validate_dataset(golden, corpus) == []


## 6. Review where error is expensive

A fixed random review percentage is not a universal rule. Prioritize critical risk, novel slices, changed generators, release-critical cases, and underrepresented slices; retain a random sample to detect unknown failure modes.


In [ ]:
risk_weight = {"low": 1, "medium": 2, "high": 4, "critical": 8}
slice_counts = Counter(c.slice for c in golden)
review_queue = sorted(golden, key=lambda c: (
    risk_weight[c.risk] + 3 / slice_counts[c.slice], c.case_id
), reverse=True)
display(pd.DataFrame([c.model_dump() for c in review_queue[:10]])[["case_id","slice","risk","review_status"]])


## 7. Promotion and versioning

The committed artifacts make CI reproducible. Promotion requires schema validity, corpus references, reviewer rationale, and an explicit review state.


In [ ]:
paths = [DATA_DIR / "evaluation_candidates.json", DATA_DIR / "evaluation_golden.json"]
for path in paths:
    print(path.name, path.stat().st_size, "bytes")
assert all(c.review_status == "approved" and c.reviewer_rationale for c in golden)
print(pd.Series(c.slice for c in golden).value_counts().to_string())


## Exercises and production upgrades

1. Add a multilingual paraphrase and document the reviewer expertise required.
2. Add a changed-answer mutation for R17 applicability and prove the old reference is rejected.
3. Create a held-out release set that generation prompts never see.
4. Replace the optional provider while preserving `CandidateDraft` and `EvalCase`.

**Checkpoint:** a dataset is fit for a release gate only when its provenance, labels, slices, risks, versions, and review state are inspectable.
